In [1]:
from biodictionary import BioDictionary

In [2]:
from dash import Dash, html, Output, Input, State, callback
import dash_cytoscape as cyto

In [3]:
cyto.load_extra_layouts()

In [4]:
dictionary = BioDictionary()

In [75]:
import networkx as nx
from networkx.classes.filters import no_filter
from typing import List
from functools import reduce

## Things I want to do

Build a sub-graph based on some starting nodes

Add to the sub-graph iteratively, expanding by adjacent nodes

Save and read graphs to file

Click on a node to show it's properties

Add context-specific interactions to nodes (show/edit runs file templates, TDFs, workflow github link, etc.)

Uses:

- Build template runs files, for a given graph traversal, with specific data types, projects, etc. build appropriate runs files
- Parse existing runs files and attach their properties to the graph
- Use a graph traversal to build and export a biograph query for use in repl
- Query to find existing data directly
- Which data were processed with workflow uuids xyz

### BioDictionary Exploration

In [161]:
dictionary.schema['aliquot_level_maf']

{'$schema': 'http://json-schema.org/draft-04/schema#',
 'id': 'aliquot_level_maf',
 'title': 'Aliquot Level MAF',
 'type': 'object',
 'category': 'data_file',
 'namespace': 'https://gdc.cancer.gov/',
 'program': '*',
 'project': '*',
 'description': 'Data file containing the observed annotated somatic mutations from a aliquot in mutation annotation format.\n',
 'additionalProperties': False,
 'submittable': True,
 'validators': None,
 'systemProperties': ['id',
  'project_id',
  'created_datetime',
  'updated_datetime'],
 'tagProperties': ['data_type'],
 'tagBuilderConfig': {'ignoreEntries': [{'path': 'vcf2maf_workflows',
    'prop': 'workflow_type',
    'values': ['SomaticSniper']}]},
 'links': [{'name': 'vcf2maf_workflows',
   'backref': 'aliquot_level_mafs',
   'label': 'data_from',
   'target_type': 'vcf2maf_workflow',
   'multiplicity': 'one_to_one',
   'required': True}],
 'required': ['submitter_id',
  'data_format',
  'data_category',
  'data_type',
  'experimental_strategy'],


In [7]:
k

'germline_variation'

In [8]:
v.keys()

dict_keys(['$schema', 'id', 'title', 'type', 'namespace', 'category', 'program', 'project', 'description', 'additionalProperties', 'submittable', 'validators', 'systemProperties', 'tagProperties', 'links', 'uniqueKeys', 'required', 'properties'])

In [11]:
v['id']

'germline_variation'

In [15]:
v['title']

'Germline Variation'

In [24]:
# v['properties']

In [25]:
# v

In [9]:
v['links']

[{'exclusive': True,
  'required': True,
  'subgroup': [{'name': 'somatic_mutation_calling_workflows',
    'backref': 'germline_variations',
    'label': 'data_from',
    'target_type': 'somatic_mutation_calling_workflow',
    'multiplicity': 'many_to_one',
    'required': True}]}]

### Graph Building

Class should be able to spit out a cytoscape representation of a graph, make a subgraph, interrogate upstream/downstream connections from a node or nodes in subgraph. 

In [79]:
class GdcBioInvalidDataKeyException(Exception):
    def __init__(self, key: str) -> None:
        self.message = f'Data key {key} is not allowed to occur in the data dictionary'

class GdcBioNode():
    DISPLAY_STATES = set(['standard', 'fringe', 'hidden'])
    SELECTED_STATES = set(['selected', 'not-selected'])
    def __init__(self, node_id: str, label: str, data: dict) -> None:
        if 'id' in data:
            raise GdcBioInvalidDataKeyException('id')
        if 'label' in data:
            raise GdcBioInvalidDataKeyException('label')
        
        self.id = node_id
        self.data = data
        self.data['id'] = node_id
        self.data['label'] = label
        self.graph = None
        # Display properties
        self.visible = True
        self.display_classes = set()

    def set_graph(self, graph: type[GdcBioGraph]) -> None:
        self.graph = graph

    def _set_classes(self, classes: List[str]) -> None:
        # TODO: validate against DISPLAY_STATES and SELECTED_STATES
        self.display_classes = classes

    def set_standard(self) -> None:
        classes = self.display_classes - GdcBioNode.DISPLAY_STATES
        classes.add('standard')
        self._set_classes(classes)

    def set_fringe(self) -> None:
        classes = self.display_classes - GdcBioNode.DISPLAY_STATES
        classes.add('fringe')
        self._set_classes(classes)

    def set_hidden(self) -> None:
        classes = self.display_classes - GdcBioNode.DISPLAY_STATES
        classes.add('hidden')
        self._set_classes(classes)

    def set_selected(self) -> None:
        classes = self.display_classes - GdcBioNode.SELECTED_STATES
        classes.add('selected')
        self._set_classes(classes)

    def set_not_selected(self) -> None:
        classes = self.display_classes - GdcBioNode.SELECTED_STATES
        classes.add('not-selected')
        self._set_classes(classes)

    def add_class(self, class_id: str) -> None:
        self.display_classes.add(class_id)

    def remove_class(self, class_id: str) -> None:
        self.display_classes.remove(class_id)

    def cyto_repr(self) -> None:
        cyto_dict = {
            'data': self.data,
            'classes': ' '.join(self.display_classes)
        }
        return cyto_dict

        
        # data_dict = {k: v for k, v in self.data}
        # data_dict['id'] = self.node_id
        # data_dict['label'] = self.data[label_field]
        # data_dict = {
        #     'data': data_dict,
        #     'classes': ' '.join(self.classes)
        # }
        # return data_dict

class GdcDiGraph(nx.DiGraph):
    '''
    Custom method override for nx.DiGraph allowing cleaner node creation
    code and cleaner node dictionaries.

    This data structure underlies some capabilities of GdcBioGraph
    '''
    def add_node(self, node: dict) -> None:
        super().add_node(node.id, **node.data)


class GdcBioGraph():
    def __init__(self, biodictionary: type[BioDictionary]=None) -> None:
        self.graph = GdcDiGraph()
        self.display_graph = None
        self.nodes = {} # dictionary holding GdcBioNode objects
        self.edges = {} # not currently used
        if biodictionary:
            node_list, edge_list = self.get_ne_from_biodictionary(biodictionary)
            # self.graph = self.from_biodictionary(biodictionary)
            for node in node_list:
                self.add_node(node)
            self.add_edges_from_list(edge_list)

    def get_ne_from_biodictionary(self, dictionary: type[BioDictionary]) -> tuple[List[dict], List[type[GdcBioNode]]]:
        '''
        Build a graph from the biodictionary.

        Returns a GdcBioGraph object
        '''
        def traverse_links(links: list | dict, lst: list=[]) -> list[str]:
            if isinstance(links, list):
                for link in links:
                    lst = traverse_links(link, lst)
            else:
                if 'subgroup' in links:
                    lst.extend(traverse_links(links['subgroup'], lst))
                else:
                    lst.append(links['target_type'])
            return lst
        # graph = GdcDiGraph()
        node_list = []
        edge_list = []
        for key, node in dictionary.schema.items():
            node_id: str = node['id']
            label: str = node['title']
            data = {
                'title': node['title'],
                'category': node['category'],
                'description': node['description'],
                'uniqueKeys': node['uniqueKeys'],
            }
            thisnode: type[GdcBioNode] = GdcBioNode(node_id=node_id, label=label, data=data)
            thisnode.add_class(node['category'])
            thisnode.add_class('standard')
            thisnode.add_class('not-selected')
            node_list.append(thisnode)
            # self.add_node(self, thisnode)

            target_list: list[str] = traverse_links(node['links'], lst=[])
            src: str = node['id']
            edges: list[tuple[str, str]] = [(src, tgt) for tgt in target_list]
            # graph.add_edges_from(edges)
            edge_list.extend(edges)
        return node_list, edge_list

    def add_node(self, node: GdcBioNode) -> None:
        # TODO: check that node doesn't already exist
        node.set_graph(self)
        self.nodes[node.id] = node
        self.graph.add_node(node)

    def add_edge(self, edge: tuple[str, str]) -> None:
        self.graph.add_edge(edge)
    
    def add_edges_from_list(self, edge_list: List[tuple]) -> None:
        self.graph.add_edges_from(edge_list)

    def cyto_edge(self, edge: tuple[str, str]) -> dict:
        data_dict = {'data': {'source': edge[0], 'target': edge[1]}}
        return data_dict

    def get_cytoscape_elements(self) -> list[dict]:
        if self.display_graph is None:
            self.init_display_graph()
        cyto_nodes = [self.nodes[node_id].cyto_repr() for node_id in self.display_graph.nodes.keys()]
        cyto_edges = [self.cyto_edge(edge_id) for edge_id, edge_data in self.display_graph.edges.items()]
        cyto_elements = cyto_nodes + cyto_edges
        return cyto_elements

    def init_display_graph(self, node_list: List[str]=None) -> None:
        '''
        Default initialization for display_graph
        '''
        node_filter_fn = no_filter
        if node_list is not None:
            node_filter_fn = lambda node: node in node_list
        self.display_graph = nx.subgraph_view(self.graph, filter_node=node_filter_fn)

    def set_node_classes(self, node_id: str, class_list: list[str]) -> None:
        '''
        set list of classes on a node directly
        '''
        self.nodes[node_id].set_classes(class_list)

    def add_display_nodes(self, node_list: list[str]) -> None:
        '''
        Make a subgraph selection from the given node_list.
        '''
        self.display_graph = nx.subgraph_view(self.graph, lambda n: n in node_list)

    def get_standard_nodes(self) -> None:
        '''
        Get list of nodes currently marked as standard
        '''
    
    def update_fringe_display_nodes(self) -> None:
        '''
        Find nodes adjacent to currently displayed nodes.
        Add them to display_graph and set their display class to 'fringe'
        '''
        
        current_display_nodes = self.display_graph.nodes.keys()
        all_predecessors = set(reduce(
            lambda i, j: i + j, 
            [list(self.graph.predecessors(node)) for node in current_display_nodes]))
        fringe_nodes = all_predecessors - current_display_nodes
        for node_id in fringe_nodes:
            self.nodes[node_id].set_fringe()

    # def add_nodes_from_list(node_list):
    #     self.graph.add_nodes_from(node_list)

In [83]:
layout = {
    'name': 'klay',
    'klay': {
        'direction': 'UP',
        'layoutHierarchy': True,
        'spacing': 40
    }
}
stylesheet=[
    {
        'selector': 'node',
        'style': {
            'content': 'data(label)'
        }
    },
    {
        'selector': '.administrative',
        'style': {
            'background-color': '#ffb703'
        }
    },
    {
        'selector': '.biospecimen',
        'style': {
            'background-color': '#8ecae6'
        }
    },
    {
        'selector': '.analysis',
        'style': {
            'background-color': '#fb8500'
        }
    },
    {
        'selector': '.data_file',
        'style': {
            'background-color': '#023047'
        }
    },
    {
        'selector': '.notation',
        'style': {
            'background-color': '#219ebc'
        }
    },
    {
        'selector': '.standard',
        'style': {
            'background-opacity': 1.0,
            'border-width': '0px'
        }
    },
    {
        'selector': '.fringe',
        'style': {
            'border-width': '2px',
            'border-style': 'dotted',
            'background-opacity': 0.5
        }
    },
]

In [84]:
app = Dash()

# foo.subgraph([nodes list ...])
# foo.show_potential_next_nodes()
# foo.subgraph.add_next_node(node)

bg = GdcBioGraph(dictionary)
initial_nodes = [
    'program',
    'project',
    'case',
    'aliquot',
    'read_group',
]
bg.init_display_graph(initial_nodes)
bg.update_fringe_display_nodes()
elements = bg.get_cytoscape_elements()
app.layout = html.Div([
    cyto.Cytoscape(
        id='bg-layout-1',
        elements=elements,
        layout=layout,
        responsive=True,
        style={'width': '100%', 'height': '600px'},
        stylesheet=stylesheet
    ),
    html.P("Notes", id='click-output')
])

@callback(Output('click-output', 'children'), Input('bg-layout-1', 'tapNodeData'))
def displayTapNodeData(data):
    if data:
        return "Clicked on " + data['label']

if __name__ == '__main__':
    app.run(debug=True)

In [82]:
bg.get_cytoscape_elements()

[{'data': {'title': 'Project',
   'category': 'administrative',
   'description': 'Any specifically defined piece of work that is undertaken or attempted to meet a single requirement. (NCIt C47885)\n',
   'uniqueKeys': [['id'], ['code']],
   'id': 'project',
   'label': 'Project'},
  'classes': 'administrative not-selected standard'},
 {'data': {'title': 'Aliquot',
   'category': 'biospecimen',
   'description': 'Pertaining to a portion of the whole; any one of two or more samples of something, of the same volume or weight.\n',
   'uniqueKeys': [['id'], ['project_id', 'submitter_id']],
   'id': 'aliquot',
   'label': 'Aliquot'},
  'classes': 'biospecimen not-selected standard'},
 {'data': {'title': 'Read Group',
   'category': 'biospecimen',
   'description': 'Sequencing reads from one lane of an NGS experiment.\n',
   'uniqueKeys': [['id'], ['project_id', 'submitter_id']],
   'id': 'read_group',
   'label': 'Read Group'},
  'classes': 'biospecimen not-selected standard'},
 {'data': {'

In [69]:
%pdb

Automatic pdb calling has been turned OFF


In [177]:
foo = GdcBioGraph(dictionary)

In [192]:
initial_nodes = [
    'program',
    'project',
    'case',
    'aliquot',
    'read_group',
]

nx.subgraph(foo.graph, initial_nodes)

In [180]:
# foo.get_cytoscape_elements()

In [125]:
# C = nx.DiGraph()
C = GdcBioGraph()

In [126]:
A = GdcBioNode('foo', {'id': 'foo', 'label': 'Foo', 'category': 'test'})
B = GdcBioNode('foo', {'id': 'foo', 'label': 'Foo', 'category': 'test'})

In [127]:
A

In [110]:
A

In [128]:
C.add_node(A)

In [129]:
C.nodes

NodeView(('foo',))

In [130]:
C.nodes['foo']

{'id': 'foo', 'label': 'Foo', 'category': 'test'}

In [53]:
A

foo

In [54]:
B

foo

In [56]:
A is B

False

In [34]:
bg.nodes['aliquot']

{'data': {'id': 'aliquot',
  'label': 'Aliquot',
  'category': 'biospecimen',
  'catrank': 1,
  'length': 3},
 'classes': 'biospecimen'}

In [38]:
[n for n in bg.predecessors('aliquot')]

['read_group', 'raw_methylation_array']

In [33]:
for key, node in dictionary.schema.items():
    break

In [34]:
node['id']

'germline_variation'

In [70]:
def traverse_links(links, lst=[]):
    if isinstance(links, list):
        for link in links:
            lst = traverse_links(link, lst)
    else:
        if 'subgroup' in links:
            lst.extend(traverse_links(links['subgroup'], lst))
        else:
            lst.append(links['target_type'])
    return lst

cat_rank = {
    'administrative': 0,
    'biospecimen': 1,
    'data_file': 2,
    'analysis': 3,
    'notation': 2
}

def build_graph(dictionary, graph=None):
    if graph is None:
        bg = nx.DiGraph()
    else:
        bg = graph
    nodes = []
    edges = []
    for key, node in dictionary.schema.items():
        thisnode = {
            'data': {
                'id': node['id'],
                'label': node['title'],
                'category': node['category'],
                'catrank' : cat_rank[node['category']]
            },
            'classes': node['category']
        }
        nodes += [(node['id'], thisnode)]
        target_list = traverse_links(node['links'], lst=[])
        src = node['id']
        edges += [(src, tgt, {'data': {'source': src, 'target': tgt}}) for tgt in target_list]
    bg.add_nodes_from(nodes)
    bg.add_edges_from(edges)
    return bg

def get_cytoscape_elements(graph):
    nodes = [node[1] for node in graph.nodes.items()]
    edges = [edge[-1] for edge in graph.edges.items()]
    elements = nodes + edges
    return elements

In [71]:
def klay_layout_graph(graph):
    elements = get_cytoscape_elements(graph)
    return html.Div([
        cyto.Cytoscape(
            id='bg-layout-1',
            elements=elements,
            layout={
                'name': 'klay',
                'klay': {
                    'direction': 'UP',
                    'layoutHierarchy': True,
                    'spacing': 40
                }
            },
            responsive=True,
            style={'width': '100%', 'height': '800px'},
            stylesheet=[
                {
                    'selector': 'node',
                    'style': {
                        'content': 'data(label)'
                    }
                },
                {
                    'selector': '[category = "administrative"]',
                    'style': {
                        'background-color': '#ffb703'
                    }
                },
                {
                    'selector': '[category = "biospecimen"]',
                    'style': {
                        'background-color': '#8ecae6'
                    }
                },
                {
                    'selector': '[category = "analysis"]',
                    'style': {
                        'background-color': '#fb8500'
                    }
                },
                {
                    'selector': '[category = "data_file"]',
                    'style': {
                        'background-color': '#023047'
                    }
                },
                {
                    'selector': '[category = "notation"]',
                    'style': {
                        'background-color': '#219ebc'
                    }
                },
            ]
        )
    ])

In [72]:
bg = build_graph(dictionary)
for node in bg.nodes.items():
    length = len(nx.shortest_path(bg, node[0], 'program')) - 1
    bg.nodes[node[0]]['data']['length'] = length


In [50]:
svg = nx.subgraph_view(bg, filter_node=no_filter)

In [51]:
svg.nodes

NodeView(('germline_variation', 'copy_number_estimate_liftover', 'copy_number_segment', 'copy_number_variation_pair_workflow', 'filtering_workflow_archive', 'absolute_auto_model_extraction_workflow', 'msi_status', 'filtered_somatic_mutation_index', 'structural_variation', 'structural_variation_liftover', 'harmonization_metric', 'submitted_unaligned_reads', 'mrvan_annotation_metric', 'project', 'tumor_mutation_burden', 'secondary_expression_analysis', 'bamqc_metric', 'copy_number_estimate', 'masked_methylation_array', 'annotation_metric', 'ensemble_raw_aliquot_maf', 'vcf2maf_workflow', 'absolute_copy_number_analysis_workflow', 'aliquot', 'raw_somatic_mutation', 'tumor_realigned_reads_index', 'annotated_somatic_mutation', 'maf_rna_annotation_workflow', 'raw_somatic_mutation_index', 'structural_variant_calling_workflow', 'somatic_mutation_calling_metric', 'mrvan_annotated_maf', 'read_group', 'somatic_mutation_calling_workflow', 'somatic_mutation_filtering_metric', 'aligned_reads_index', '

In [30]:
initial_nodes = [
    'program',
    'project',
    'case',
    'aliquot',
    'read_group',
]
sgv = nx.subgraph_view(bg, filter_node=lambda n: n in initial_nodes)

In [28]:
# Subgraph view updates after it's creation
# bg = nx.DiGraph()
# sgv = nx.subgraph_view(bg)
# bg = build_graph(dictionary, bg)

In [31]:
sgv.nodes

NodeView(('project', 'aliquot', 'read_group', 'case', 'program'))

In [36]:
list(sgv.predecessors('aliquot'))

['read_group']

In [37]:
list(sgv.successors('aliquot'))

['case']

In [48]:
from itertools import accumulate
from functools import reduce

In [53]:
set(reduce(lambda i, j: i + j, [list(sgv.predecessors(node)) for node in initial_nodes]))

{'aliquot', 'case', 'project', 'read_group'}

In [ ]:
def get_reachable_nodes(in_nodes = []):
    in_nodes = 
    reachable_nodes = [list(sgv.prececessors(node)) for node in initial_nodes]

In [ ]:
reduce(
    lambda i, j: i + j,
    set(sgv.predecesors(node))
)

In [68]:
app = Dash()
app.layout = klay_layout_graph(sgv)

if __name__ == '__main__':
    app.run(debug=True)

NameError: name 'sgv' is not defined

> /tmp/ipykernel_344958/3175660443.py(2)<module>()
      1 app = Dash()
----> 2 app.layout = klay_layout_graph(sgv)
      3 
      4 if __name__ == '__main__':
      5     app.run(debug=True)



ipdb>  exit


In [74]:
initial_nodes = [
    'program',
    'project',
    'case',
    'aliquot',
    'read_group',
]
sg = nx.subgraph(bg, initial_nodes)

app = Dash()
app.layout = klay_layout_graph(sg)

if __name__ == '__main__':
    app.run(debug=True)

## Notes

Using dash-cytoscape I can use cytoscape to plot the biograph or part thereof and enable interactive callbacks

Next steps

- Plot subgraphs
- Interactively build a data path starting from Aliquot
  - End node should show possible downstream nodes
  - Click on a downstream node to select it and set it as the new end node
  - Layout should update and animate as nodes are added
- Read TDFs and Runs files 

In [10]:
app = Dash()

app.layout = html.Div([
    cyto.Cytoscape(
        id='bg-layout-1',
        elements=elements,
        # layout={
        #     'name': 'klay',
        #     'klay': {
        #         'direction': 'UP',
        #         'layoutHierarchy': True,
        #         'spacing': 40
        #     }
        # },
        # layout={
        #     'name': 'breadthfirst',
        #     'roots': ["program"],
        #     'animate': True,
        #     # 'grid': True,
        #     'padding': 60
        # },
        layout={
            'name': 'cola',
            'maxSimulationTime': 20000,
            'flow': { 'axis': 'y', 'minSeparation': 30 }
        },
        responsive=True,
        style={'width': '100%', 'height': '800px'},
        stylesheet=[
            {
                'selector': 'node',
                'style': {
                    'content': 'data(label)'
                }
            },
            {
                'selector': '[category = "administrative"]',
                'style': {
                    'background-color': '#ffb703'
                }
            },
            {
                'selector': '[category = "biospecimen"]',
                'style': {
                    'background-color': '#8ecae6'
                }
            },
            {
                'selector': '[category = "analysis"]',
                'style': {
                    'background-color': '#fb8500'
                }
            },
            {
                'selector': '[category = "data_file"]',
                'style': {
                    'background-color': '#023047'
                }
            },
            {
                'selector': '[category = "notation"]',
                'style': {
                    'background-color': '#219ebc'
                }
            },
        ]
    )
])

# app.clientside_callback(
#     """
#     function (id, layout) {
#         layout.depthSort = (a, b) => b.data('length') - a.data('length');
#         cy.layout(layout).run();
#         return layout;
#     }
#     """,
#     Output('bg-layout-1', 'layout'),  # update the (dash) cytoscape component's layout
#     Input('bg-layout-1', 'id'),       # trigger the function when the Cytoscape component loads (1)
#     State('bg-layout-1', 'layout'),   # grab the layout so we can update it in the function
#     prevent_initial_call=False # ensure (1) (needed if True at the app level)
# )

if __name__ == '__main__':
    app.run(debug=True)

In [95]:
set([i['data']['category'] for i in nodes])

{'administrative', 'analysis', 'biospecimen', 'data_file', 'notation'}